# 🍽️ Menu AR — version vente (uniquement des composants utilisables commercialement)

**Réglage :** `Exécution > Modifier le type d'exécution > A100 GPU` + **RAM élevée**. Lance les cellules **1, 2, 3** puis scanne le QR code.

Ce qui change par rapport au notebook de test :
- **Rendu 3D** : notre propre rasteriseur (`menu_raster`, écrit de zéro) à la place de nvdiffrast (NVIDIA, non commercial).
- **CuMesh** : branche `vente`, où le fichier mémoire GPU de cubvh (NVIDIA, non commercial) est remplacé par le nôtre.
- **Détection assiette / sauces / sachets** : Grounding DINO + SAM 2.1 (licences Apache 2.0) à la place d'Ultralytics (AGPL).
- La **cellule 4** (facultative, hors produit) compare l'ancien et le nouveau sur tes plats.

Licences restantes : TRELLIS.2, CuMesh, FlexGEMM, o-voxel, utils3d (MIT) · MV-Adapter, LaMa, Grounding DINO, SAM 2.1 (Apache 2.0) · BiRefNet, VAE SDXL fp16 (MIT) · Real-ESRGAN (BSD) · SDXL (OpenRAIL++, usage commercial autorisé avec restrictions d'usage) · **DINOv3 (licence Meta, à faire valider par un juriste)**.


In [ ]:
# CELLULE 1 — GPU + Drive + modèles (version vente)
import os
os.environ['MENU_AR_LIBRE'] = '1'        # détecteur libre (Grounding DINO + SAM 2.1)
os.environ['MENU_AR_RASTER'] = 'libre'    # notre rasteriseur
import os, torch
from google.colab import drive
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN'
mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU : {gpu} ({mem:.0f} Go) | torch {torch.__version__}')
assert mem >= 22, "❌ Il faut un A100 : Exécution > Modifier le type d'exécution."

drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/trellis2_backup'
HF_CACHE = f'{BACKUP}/hf_cache'
SMALL = f'{BACKUP}/small_models'          # détourage / détection / segmentation (cellule 6)
if os.path.isdir(SMALL):
    os.environ['U2NET_HOME'] = f'{SMALL}/u2net'
    os.environ['MENU_AR_WEIGHTS'] = f'{SMALL}/ultralytics'
    print('✅ Petits modèles chargés depuis ton Drive')
# Modèles Hugging Face. Une copie complète sur ton Drive est lue directement ; un modèle absent ou
# incomplet est téléchargé sur le disque de la session (fiable, rapide), puis recopié sur ton Drive
# en arrière-plan. (Télécharger directement dans Drive coupait les gros fichiers : c'est ce qui avait
# empêché la texture IA de se charger.)
import shutil, threading
LOCAL = '/content/hf_cache'; os.makedirs(LOCAL, exist_ok=True)
os.environ['HF_HUB_CACHE'] = LOCAL
MODELS = {   # dépôt -> fichiers utiles (None = tout le dépôt)
    'microsoft/TRELLIS.2-4B': None, 'microsoft/TRELLIS-image-large': None,
    'facebook/dinov3-vitl16-pretrain-lvd1689m': None,
    'stabilityai/stable-diffusion-xl-base-1.0': ['model_index.json', 'scheduler/*', 'tokenizer/*', 'tokenizer_2/*',
        'text_encoder/config.json', 'text_encoder/model.fp16.safetensors', 'text_encoder_2/config.json',
        'text_encoder_2/model.fp16.safetensors', 'unet/config.json', 'unet/diffusion_pytorch_model.fp16.safetensors',
        'vae/config.json'],
    'madebyollin/sdxl-vae-fp16-fix': ['config.json', 'diffusion_pytorch_model.safetensors'],
    'huanngzh/mv-adapter': ['mvadapter_ig2mv_sdxl.safetensors'],
    'IDEA-Research/grounding-dino-base': None,       # détection par texte (Apache 2.0)
    'facebook/sam2.1-hiera-large': None,              # contours (Apache 2.0)
}
def _dir(root, repo): return f"{root}/models--{repo.replace('/', '--')}"
def drive_ok(repo):
    d = _dir(HF_CACHE, repo)
    if os.path.exists(f'{d}/.complet'): return True
    if not os.path.isdir(f'{d}/snapshots'): return False
    w = [os.path.join(r, f) for r, _, fs in os.walk(f'{d}/snapshots') for f in fs
         if f.endswith(('.safetensors', '.pt', '.pth', '.bin', '.ckpt'))]
    ok = bool(w) and all(os.path.exists(x) and os.path.getsize(os.path.realpath(x)) > 1_000_000 for x in w)
    if ok: open(f'{d}/.complet', 'w').close()
    return ok
missing = []
for repo in MODELS:
    loc = _dir(LOCAL, repo)
    if os.path.isdir(HF_CACHE) and drive_ok(repo):
        if not os.path.exists(loc): os.symlink(_dir(HF_CACHE, repo), loc)
    elif not os.path.isdir(loc) or os.path.islink(loc):
        missing.append(repo)
if missing:
    print('⏳ À télécharger une fois (puis gardés sur ton Drive) :', ', '.join(missing))
    from huggingface_hub import login, snapshot_download
    try:
        from google.colab import userdata; login(userdata.get('HF_TOKEN')); print('✅ Jeton Hugging Face OK')
    except Exception:
        if any('dinov3' in m for m in missing):
            raise SystemExit('❌ DINOv3 doit être téléchargé et le secret HF_TOKEN est absent.')
    for repo in missing:
        loc = _dir(LOCAL, repo)
        if os.path.islink(loc): os.unlink(loc)
        snapshot_download(repo, allow_patterns=MODELS[repo], cache_dir=LOCAL)
        print('  ✅', repo)
    def _to_drive(repos):     # copie sur ton Drive : refs + fichiers réels (sans liens), marqueur à la fin
        for repo in repos:
            src, dst = _dir(LOCAL, repo), _dir(HF_CACHE, repo)
            tmp = dst + '.copie'
            shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
            shutil.copytree(f'{src}/refs', f'{tmp}/refs')
            shutil.copytree(f'{src}/snapshots', f'{tmp}/snapshots', symlinks=False)
            open(f'{tmp}/.complet', 'w').close()
            shutil.rmtree(dst, ignore_errors=True); os.rename(tmp, dst)
            print(f'💾 {repo} sauvegardé sur ton Drive')
    if os.path.isdir(BACKUP):
        os.makedirs(HF_CACHE, exist_ok=True)
        threading.Thread(target=_to_drive, args=(missing,), daemon=True).start()
        print('💾 Copie vers ton Drive en arrière-plan (garde la session ouverte quelques minutes)')
os.environ['HF_HUB_OFFLINE'] = '1'
print('✅ Modèles prêts (lus sans Hugging Face)')


In [ ]:
# CELLULE 2 — Installation (branche « vente » de ton GitHub ; 15-25 min la 1re fois, ~1 min ensuite)
import os, sys, subprocess, glob, re, torch

def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        errs = [l for l in (r.stdout + r.stderr).splitlines() if re.search(r'error|fatal|killed', l, re.I) and 'Werror' not in l]
        print('\n'.join(dict.fromkeys(errs[:60])) or (r.stdout + r.stderr)[-3000:])
        raise SystemExit(f'❌ Échec : {cmd[:120]} — copie-moi les lignes ci-dessus.')
    return r.stdout

sh('pip uninstall -y -q tensorflow tf-keras tensorflow-hub || true')
if not os.path.exists('/content/TRELLIS.2'):
    sh('git clone -q -b vente https://github.com/Omazi/TRELLIS.2.git /content/TRELLIS.2')
else:   # toujours la dernière version de la branche vente (même si un fichier a été modifié ici)
    sh('git -C /content/TRELLIS.2 fetch -q origin vente && git -C /content/TRELLIS.2 checkout -q -B vente origin/vente '
       '&& git -C /content/TRELLIS.2 reset -q --hard origin/vente')
CODE_REV = sh('git -C /content/TRELLIS.2 rev-parse --short HEAD').strip()
# Code mis à jour dans une session déjà lancée : on oublie l'ancien code Python chargé
_ours = ('trellis2.menu_ar_pipeline', 'trellis2.utils.dish_segmentation', 'trellis2.utils.photo_projection',
         'trellis2.utils.food_postprocess', 'trellis2.utils.mv_texture', 'mvadapter', 'webapp', 'menu_raster')
for _m in [m for m in sys.modules if m.startswith(_ours)]:
    del sys.modules[_m]
print('✅ Code', CODE_REV)
sh('pip install -q easydict plyfile zstandard kornia timm imageio imageio-ffmpeg trimesh rembg onnxruntime ninja scikit-image '
   'diffusers peft accelerate spandrel==0.4.2 gltflib jaxtyping typeguard omegaconf einops fastapi uvicorn python-multipart qrcode')
sh('pip install -q /content/TRELLIS.2/third_party/utils3d')   # copie dans ton dépôt
import site   # notre rasteriseur (menu_raster) visible de partout, y compris des paquets installés
open(os.path.join(site.getsitepackages()[0], 'menu_ar.pth'), 'w').write('/content/TRELLIS.2\n')
sys.path.insert(0, '/content/TRELLIS.2')
# Node.js >= 18 pour l'optimisation mobile du GLB
nv = subprocess.run('node -v', shell=True, text=True, capture_output=True).stdout.strip()
if not nv or int(nv.lstrip('v').split('.')[0]) < 18:
    sh('curl -fsSL https://nodejs.org/dist/v22.11.0/node-v22.11.0-linux-x64.tar.xz | tar -xJ -C /usr/local --strip-components=1')
sh('cd /content/TRELLIS.2/tools/mobile_glb && npm install -q --no-audit --no-fund')
print('✅ Node', subprocess.run('node -v', shell=True, text=True, capture_output=True).stdout.strip())

# Toolkit CUDA de même version majeure que PyTorch
tcuda = torch.version.cuda
def nvcc_ver(home):
    out = subprocess.run(f'{home}/bin/nvcc --version', shell=True, text=True, capture_output=True).stdout
    m = re.search(r'release (\d+\.\d+)', out); return m.group(1) if m else ''
match = [h for h in sorted(glob.glob('/usr/local/cuda-*')) + ['/usr/local/cuda'] if nvcc_ver(h).split('.')[0] == tcuda.split('.')[0]]
if not match:
    maj, mi = tcuda.split('.')
    subprocess.run(f'apt-get -qq update && apt-get -qq install -y cuda-toolkit-{maj}-{mi}', shell=True)
    match = [h for h in sorted(glob.glob('/usr/local/cuda-*')) if nvcc_ver(h).split('.')[0] == maj]
    if not match: raise SystemExit(f'❌ Toolkit CUDA {tcuda} introuvable.')
os.environ['CUDA_HOME'] = match[-1]
os.environ['PATH'] = f'{match[-1]}/bin:' + os.environ['PATH']
cap = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
os.environ['MAX_JOBS'] = str(os.cpu_count())

# Composants GPU : wheels sauvegardés sur Drive, sinon compilation
tag = f'torch{torch.__version__.replace("+","_")}_py{sys.version_info[0]}{sys.version_info[1]}_sm{cap[0]}{cap[1]}'
cache = f'/content/drive/MyDrive/trellis2_wheels/{tag}' if os.path.ismount('/content/drive') else f'/content/wheels/{tag}'
os.makedirs(cache, exist_ok=True)
# Chaque composant est recompilé quand SA source change (clé = version exacte du code)
def _remote(repo, branch):
    return sh(f'git ls-remote https://github.com/Omazi/{repo}.git {branch}').split()[0][:10]
exts = {   # nom : (clé de version, commande pour récupérer la source) — plus de nvdiffrast
    'cumesh':    (_remote('CuMesh', 'vente'), 'git clone -q -b vente https://github.com/Omazi/CuMesh.git /tmp/ext/cumesh'),
    'flex_gemm': (_remote('FlexGEMM', 'main'), 'git clone -q --recursive https://github.com/Omazi/FlexGEMM.git /tmp/ext/flex_gemm'),
    'o_voxel':   (sh('git -C /content/TRELLIS.2 rev-parse HEAD:o-voxel').strip()[:10], 'cp -r /content/TRELLIS.2/o-voxel /tmp/ext/o_voxel'),
}
os.makedirs('/tmp/ext', exist_ok=True)
sh('pip uninstall -y -q nvdiffrast || true')     # s'assurer qu'aucun ancien nvdiffrast ne traîne
for name, (key, get) in exts.items():
    d = f'{cache}/vente/{name}-{key}'; os.makedirs(d, exist_ok=True)
    found = glob.glob(f'{d}/*.whl')
    if found:
        sh(f'pip install -q --no-deps --force-reinstall "{found[0]}"'); print(f'✅ {name} {key} (déjà compilé)'); continue
    print(f'🔨 Compilation de {name} ({key})...')
    sh(f'rm -rf /tmp/ext/{name}'); sh(get)
    sh(f'pip wheel -v --no-build-isolation --no-deps /tmp/ext/{name} -w {d}')
    whl = glob.glob(f'{d}/*.whl')[0]
    sh(f'pip install -q --no-deps --force-reinstall "{whl}"'); print(f'✅ {name} compilé')
print('🎉 Installation terminée')

In [ ]:
# CELLULE 3 — Lancer le site (version vente) : photo -> 3D -> AR depuis ton téléphone
# Lance d'abord les cellules 1 et 2.
# Le site tourne tant que cette session Colab reste ouverte.
import os, sys, re, time, secrets, subprocess, io
sys.path.insert(0, '/content/TRELLIS.2'); os.chdir('/content/TRELLIS.2')
from IPython.display import display, HTML
from trellis2.menu_ar_pipeline import MenuARPipeline
from webapp.server import create_app, serve

import trellis2.menu_ar_pipeline as _map
# 1. pipeline : reconstruit si le code a changé (sinon le site tournerait avec l'ancienne version)
if globals().get('MENU_REV') != _map.CODE_VERSION:
    import gc, torch
    for _v in ('menu_pipe', 'mv_tex', 'auto_seg'):
        globals().pop(_v, None)
    gc.collect(); torch.cuda.empty_cache()
    menu_pipe = MenuARPipeline(trellis_pipeline=globals().get('pipeline'))   # TRELLIS gardé, le reste rechargé
    pipeline, auto_seg, mv_tex = menu_pipe.trellis, menu_pipe.segmenter, menu_pipe.texturer
    MENU_REV = _map.CODE_VERSION
print('✅ Pipeline version', MENU_REV, '(vente) | texture IA :', 'oui' if menu_pipe.texturer else 'NON (voir erreur ci-dessus)',
      '| détecteur :', type(menu_pipe.segmenter).__name__)
import importlib.util
assert importlib.util.find_spec('ultralytics') is None and importlib.util.find_spec('nvdiffrast') is None, \
    '❌ Un composant non commercial est installé dans cette session : Redémarrer la session.'
print('✅ Aucun composant non commercial chargé (ni nvdiffrast, ni Ultralytics)')
def site_job(*a, **k):          # le site appelle toujours le pipeline le plus récent
    return globals()['menu_pipe'].run(*a, **k)

# 2. serveur web (plats gardés sur ton Drive s'il est monté)
DATA = '/content/drive/MyDrive/menu_ar_plats' if os.path.ismount('/content/drive') else '/content/menu_ar_plats'
if 'SITE_KEY' in globals() and 'SITE_V15' not in globals():
    raise SystemExit('❌ Un ancien site tourne encore : Exécution > Redémarrer la session, puis cellules 1, 2 et 3.')
if 'SITE_KEY' not in globals():
    SITE_KEY = secrets.token_hex(6); SITE_V15 = True
    serve(create_app(site_job, DATA, SITE_KEY), port=8000)

# 3. tunnel HTTPS public (Cloudflare, sans compte) pour l'ouvrir sur le téléphone
if 'SITE_URL' not in globals():
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.run('curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)
    tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000'],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    SITE_URL = None; t0 = time.time()
    while time.time() - t0 < 60 and not SITE_URL:
        line = tunnel.stdout.readline()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line or '')
        if m: SITE_URL = m.group(0)
    if not SITE_URL:
        raise SystemExit('❌ Le tunnel ne démarre pas. Relance la cellule.')
    time.sleep(3)   # le temps que l'adresse soit joignable

link = f'{SITE_URL}/?k={SITE_KEY}'
import qrcode
qr = qrcode.make(link); b = io.BytesIO(); qr.save(b, 'PNG')
import base64
display(HTML(f'''<div style="font-family:system-ui;display:flex;gap:20px;align-items:center">
  <img src="data:image/png;base64,{base64.b64encode(b.getvalue()).decode()}" width="190">
  <div><div style="font-size:18px;font-weight:600;margin-bottom:6px">📱 Scanne avec ton téléphone</div>
  <a href="{link}" target="_blank" style="font-size:14px;word-break:break-all">{link}</a>
  <p style="color:#666;font-size:13px;margin-top:8px">Garde ce lien pour toi : la clé à la fin protège ton GPU.<br>
  Les plats générés sont enregistrés dans {DATA}.</p></div></div>'''))

In [ ]:
# CELLULE 4 — COMPARAISON (facultative, hors produit) : ancien vs nouveau sur tes plats
# Installe TEMPORAIREMENT nvdiffrast et Ultralytics dans cette session, uniquement pour comparer.
# Après cette cellule : Exécution > Redémarrer la session avant de relancer le site (cellule 3).
import os, sys, glob, json, subprocess, time, numpy as np, torch, trimesh
from PIL import Image
from IPython.display import display
sys.path.insert(0, '/content/TRELLIS.2'); sys.path.insert(0, '/content/TRELLIS.2/third_party/mvadapter')
PLATS = '/content/drive/MyDrive/menu_ar_plats'
jobs = sorted([d for d in glob.glob(f'{PLATS}/*') if os.path.exists(f'{d}/plat_hd.glb')], key=os.path.getmtime)[-4:]
assert jobs, '❌ Aucun plat sur ton Drive : génère d’abord un plat avec le site.'

# --- 1. installer nvdiffrast (wheel déjà compilé par le notebook de test, sinon compilation) ---
tag = f'torch{torch.__version__.replace("+","_")}_py{sys.version_info[0]}{sys.version_info[1]}_sm{"".join(map(str, torch.cuda.get_device_capability()))}'
whl = glob.glob(f'/content/drive/MyDrive/trellis2_wheels/{tag}/nvdiffrast*.whl')
if whl:
    subprocess.run(f'pip install -q --no-deps "{whl[0]}"', shell=True, check=True)
else:
    print('🔨 Compilation de nvdiffrast pour la comparaison (une fois)...')
    subprocess.run('rm -rf /tmp/nvd && git clone -q --depth 1 https://github.com/Omazi/TRELLIS.2.git /tmp/nvd && '
                   'pip install -q --no-build-isolation /tmp/nvd/third_party/nvdiffrast', shell=True, check=True)
import nvdiffrast.torch as nvdr
import menu_raster as mr

def compare(pos, tri, res, label):
    a, _ = nvdr.rasterize(nvdr.RasterizeCudaContext(), pos, tri, res)
    b, _ = mr.rasterize(None, pos, tri, res)
    cov_a, cov_b = a[..., 3] > 0, b[..., 3] > 0
    same = (a[..., 3] == b[..., 3]) & cov_a
    diff = (a[..., :3] - b[..., :3]).abs()[same].max().item() if same.any() else 0.0
    agree = same.sum().item() / max(1, (cov_a | cov_b).sum().item())
    ok = agree > 0.999 and diff < 1e-3
    print(f'  {label:28s} pixels identiques {agree:8.4%} | écart max (u, v, z) {diff:.1e} | {"✅" if ok else "❌"}')
    return ok

# --- 2. rasteriseur : sur chaque plat, la texture (espace UV 4096²) et les 6 vues de MV-Adapter ---
from mvadapter.utils.mesh_utils import get_orthogonal_camera
from mvadapter.utils.mesh_utils.utils import get_clip_space_position
import trellis2.utils.mv_texture as mt
verdicts = []
print('Rasteriseur : nvdiffrast (NVIDIA) vs menu_raster (le nôtre)')
for j in jobs:
    g = list(trimesh.load(f'{j}/plat_hd.glb').geometry.values())[0]
    V = torch.tensor(np.asarray(g.vertices), dtype=torch.float32, device='cuda')
    F = torch.tensor(np.asarray(g.faces), dtype=torch.int32, device='cuda')
    UV = torch.tensor(np.asarray(g.visual.uv), dtype=torch.float32, device='cuda')
    print(f'Plat {os.path.basename(j)} ({len(F)} triangles)')
    pos = torch.cat([UV * 2 - 1, torch.zeros_like(UV[:, :1]), torch.ones_like(UV[:, :1])], 1)[None]
    verdicts.append(compare(pos, F, (4096, 4096), 'texture 4096² (o-voxel)'))
    Vn = V - (V.min(0).values + V.max(0).values) / 2; Vn = Vn / Vn.abs().max() * 0.5
    cams = get_orthogonal_camera(elevation_deg=mt.VIEW_ELEVATIONS, distance=[1.8] * 6, left=-0.55, right=0.55,
                                 bottom=-0.55, top=0.55, azimuth_deg=[x - 90 for x in mt.VIEW_AZIMUTHS], device='cuda')
    clip = get_clip_space_position(Vn, cams.mvp_mtx)
    verdicts.append(compare(clip.contiguous(), F, (768, 768), '6 vues 768² (MV-Adapter)'))
print('➡️  Rasteriseur :', 'IDENTIQUE à nvdiffrast ✅' if all(verdicts) else 'DIFFÉRENT ❌ (copie-moi ce qui précède)')

# --- 3. détourage : Ultralytics YOLOE (ancien) vs Grounding DINO + SAM 2.1 (nouveau) ---
subprocess.run('pip install -q ultralytics git+https://github.com/ultralytics/CLIP.git', shell=True, check=True)
from trellis2.utils.dish_segmentation import AutoDishSegmenter, OpenDishSegmenter
old, new = AutoDishSegmenter(), (globals().get('auto_seg') if type(globals().get('auto_seg')).__name__ == 'OpenDishSegmenter' else OpenDishSegmenter())
def show(rgba):
    bg = Image.new('RGBA', rgba.size, (70, 170, 70, 255)); bg.alpha_composite(rgba.convert('RGBA'))
    im = bg.convert('RGB'); im.thumbnail((360, 360)); return im
photos = [p for j in jobs for p in sorted(glob.glob(f'{j}/photo*.jpg'))][:6]
print('\nDétourage (vert = retiré) — à gauche : ancien (Ultralytics), à droite : nouveau (libre)')
for p in photos:
    img = Image.open(p).convert('RGB')
    a = show(old(img)); fa = list(old.last_found)
    b = show(new(img)); fb = list(new.last_found)
    row = Image.new('RGB', (a.width + b.width + 10, max(a.height, b.height)), 'white'); row.paste(a, (0, 0)); row.paste(b, (a.width + 10, 0))
    display(row)
    print(f'  ancien : {", ".join(fa) or "rien"}  |  nouveau : {", ".join(fb) or "rien"}')
print('\n⚠️ Comparaison terminée : Exécution > Redémarrer la session avant de relancer le site (cellule 3).')
